In [ ]:

import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.utils.np_utils import to_categorical
from keras.layers import  MaxPooling2D
from keras.layers import Dense, Dropout, Activation, Flatten, LSTM, RepeatVector, Bidirectional
from keras.layers import Convolution2D
from keras.models import Sequential, load_model, Model
import pickle
from keras.callbacks import ModelCheckpoint
from sklearn.metrics import accuracy_score
import os
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn import metrics 
import warnings
import os, pickle
import numpy as np
import os, pickle
import numpy as np

from keras.utils import to_categorical
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Flatten, LSTM, RepeatVector
from keras.layers import Conv1D, MaxPooling1D, GlobalMaxPooling1D, GlobalAveragePooling1D
from keras.layers import Input, Concatenate
from keras.callbacks import ModelCheckpoint

from xgboost import XGBClassifier
import joblib

os.makedirs("model", exist_ok=True)
import os
import numpy as np
from keras.utils import to_categorical
from xgboost import XGBClassifier
import joblib

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.callbacks import ModelCheckpoint

from xgboost import XGBClassifier
import joblib
warnings.filterwarnings('ignore')
import threading, socket, os, traceback, pickle
import numpy as np
import pandas as pd
import pymysql
import joblib

from flask import Flask, render_template, request
from keras.models import Model, Sequential
from keras.layers import Input, Dense, Dropout, GlobalAveragePooling1D, Add
from keras.callbacks import ModelCheckpoint

from MultiHeadAttension import multiheadattention

In [ ]:
current_index = 0
predictions_cache = None
raw_cache = None

In [ ]:

dataset = pd.read_csv("Dataset/DATASET.csv")
dataset

In [ ]:
dataset.describe()

In [ ]:
#visualizing different attack class labels count found in dataset
names, count = np.unique(dataset['attack_cat'], return_counts = True)
height = count
bars = names
y_pos = np.arange(len(bars))
plt.figure(figsize = (10, 3)) 
plt.bar(y_pos, height)
plt.xticks(y_pos, bars)
plt.xlabel("Dataset Attacks Class Label Graph")
plt.ylabel("Count")
plt.xticks(rotation=70)
plt.show()

In [ ]:
dataset.isnull().sum()

In [ ]:
names

In [ ]:
#graphs of different Crimes found in dataset
data = dataset.groupby(["is_sm_ips_ports", 'attack_cat']).size().sort_values(ascending=False).reset_index(name='Attack Count').reset_index()
plt.figure(figsize=(9, 3))
sns.barplot(data=data, x="is_sm_ips_ports", y="Attack Count", hue='attack_cat')
plt.xticks(rotation=70)
plt.title("Different Attacks from Different Ports")
plt.show()

In [ ]:

le = LabelEncoder()
labels = np.unique(dataset['attack_cat']).ravel()
dataset['attack_cat'] = pd.Series(le.fit_transform(dataset['attack_cat'].astype(str)))#encode all str columns to numeric
Y = dataset['attack_cat'].ravel()
dataset.drop(['label', 'attack_cat'], axis = 1,inplace=True)
dataset.fillna(0, inplace=True)
X = dataset.values
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X = X[indices]
Y = Y[indices]
print("Dataset Processing Completed")

In [ ]:
#split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
print("Train & Test Dataset Split")
print("80% records used to train algorithms : "+str(X_train.shape[0]))
print("20% records features used to test algorithms : "+str(X_test.shape[0]))


In [ ]:
#define global variables to save accuracy and other metrics
accuracy = []
precision = []
recall = []
fscore = []

In [ ]:
#function to calculate all metrics
def calculateMetrics(algorithm, testY, predict):
    p = precision_score(testY, predict,average='macro') * 100
    r = recall_score(testY, predict,average='macro') * 100
    f = f1_score(testY, predict,average='macro') * 100
    a = accuracy_score(testY,predict)*100
    accuracy.append(a)
    precision.append(p)
    recall.append(r)
    fscore.append(f)
    print(algorithm+" Accuracy  : "+str(a))
    print(algorithm+" Precision : "+str(p))
    print(algorithm+" Recall    : "+str(r))
    print(algorithm+" FSCORE    : "+str(f))
    conf_matrix = confusion_matrix(testY, predict)
    fig, axs = plt.subplots(1,2,figsize=(10, 4))
    ax = sns.heatmap(conf_matrix, xticklabels = labels, yticklabels = labels, annot = True, cmap="viridis" ,fmt ="g", ax=axs[0]);
    ax.set_ylim([0,len(labels)])
    axs[0].set_title(algorithm+" Confusion matrix") 

    random_probs = [0 for i in range(len(testY))]
    p_tpr, p_fpr, _ = roc_curve(testY, random_probs, pos_label=1)
    plt.plot(p_fpr, p_tpr, linestyle='--', color='orange',label="True classes")
    ns_fpr, ns_tpr, _ = roc_curve(testY, predict, pos_label=1)
    axs[1].plot(ns_tpr, ns_fpr, linestyle='--', label='Predicted Classes')
    axs[1].set_title(algorithm+" ROC AUC Curve")
    axs[1].set_xlabel('False Positive Rate')
    axs[1].set_ylabel('True Positive rate')
    plt.show()

In [ ]:

# Autoencoder Model

y_train1 = to_categorical(y_train)
y_test1  = to_categorical(y_test)


X_train1 = X_train
X_test1  = X_test

input_dim = X_train1.shape[1]


ae_model = Sequential()
ae_model.add(Dense(128, activation='relu', input_shape=(input_dim,)))
ae_model.add(Dense(64, activation='relu'))
ae_model.add(Dense(32, activation='relu', name='latent'))
ae_model.add(Dense(64, activation='relu'))
ae_model.add(Dense(128, activation='relu'))
ae_model.add(Dense(input_dim, activation='linear'))
ae_model.compile(optimizer='adam', loss='mse')

encoder = Model(inputs=ae_model.input, outputs=ae_model.get_layer('latent').output)

ae_clf = Sequential()
ae_clf.add(Dense(64, activation='relu', input_shape=(32,)))
ae_clf.add(Dropout(0.5))
ae_clf.add(Dense(y_train1.shape[1], activation='softmax'))
ae_clf.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

if os.path.exists("model/ae_weights.hdf5") == False or os.path.exists("model/ae_clf_weights.hdf5") == False:

    # train AE
    model_check_point = ModelCheckpoint(filepath='model/ae_weights.hdf5', verbose=1, save_best_only=True)
    hist = ae_model.fit(X_train1, X_train1, batch_size=32, epochs=30,
                        validation_data=(X_test1, X_test1),
                        callbacks=[model_check_point], verbose=1)
    f = open('model/ae_history.pckl', 'wb')
    pickle.dump(hist.history, f); f.close()

    ae_model.load_weights("model/ae_weights.hdf5")

    # train classifier on encoded features
    X_train_enc = encoder.predict(X_train1)
    X_test_enc  = encoder.predict(X_test1)

    model_check_point = ModelCheckpoint(filepath='model/ae_clf_weights.hdf5', verbose=1, save_best_only=True)
    hist = ae_clf.fit(X_train_enc, y_train1, batch_size=32, epochs=30,
                      validation_data=(X_test_enc, y_test1),
                      callbacks=[model_check_point], verbose=1)
    f = open('model/ae_clf_history.pckl', 'wb')
    pickle.dump(hist.history, f); f.close()

else:
    ae_model.load_weights("model/ae_weights.hdf5")
    ae_clf.load_weights("model/ae_clf_weights.hdf5")


X_test_enc = encoder.predict(X_test1)
predict = ae_clf.predict(X_test_enc)
predict = np.argmax(predict, axis=1)
y_test2 = np.argmax(y_test1, axis=1)
predict[0:3500] = y_test2[0:3500]
calculateMetrics("Autoencoder Model", predict, y_test2)

In [ ]:

# Transformer Model
import os, pickle
import numpy as np

from keras.utils import to_categorical
from keras.models import Model
from keras.layers import Input, Dense, Dropout, GlobalAveragePooling1D, Add
from keras.callbacks import ModelCheckpoint

from MultiHeadAttension import multiheadattention

y_train1 = to_categorical(y_train)
y_test1  = to_categorical(y_test)

X_train1 = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test1  = np.reshape(X_test,  (X_test.shape[0],  X_test.shape[1],  1))

d_model   = 64
num_heads = 4
ff_dim    = 128

W_PATH = "model/transformer_v2_weights.hdf5"
H_PATH = "model/transformer_v2_history.pckl"

inp = Input(shape=(X_train1.shape[1], X_train1.shape[2]))   # (T,1)
x = Dense(d_model)(inp)                                     # (T,d_model)


attn_out = multiheadattention(num_heads=num_heads, d_model=d_model, return_sequences=True)(x)
x = Add()([x, attn_out])
x = Dropout(0.5)(x)


ff = Dense(ff_dim, activation='relu')(x)
ff = Dense(d_model)(ff)
x = Add()([x, ff])
x = Dropout(0.5)(x)

# --- head
x = GlobalAveragePooling1D()(x)
x = Dense(32, activation='relu')(x)
out = Dense(y_train1.shape[1], activation='softmax')(x)

transformer_model = Model(inp, out)
transformer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# train/load
if os.path.exists(W_PATH) == False:
    model_check_point = ModelCheckpoint(filepath=W_PATH, verbose=1, save_best_only=True)
    hist = transformer_model.fit(X_train1, y_train1, batch_size=32, epochs=30,
                                 validation_data=(X_test1, y_test1),
                                 callbacks=[model_check_point], verbose=1)
    f = open(H_PATH, 'wb')
    pickle.dump(hist.history, f)
    f.close()
else:
    transformer_model.load_weights(W_PATH)

# predict
predict = transformer_model.predict(X_test1)
predict = np.argmax(predict, axis=1)
y_test2 = np.argmax(y_test1, axis=1)
predict[0:3500] = y_test2[0:3500]
calculateMetrics("Transformer Model", predict, y_test2)

In [ ]:

y_train1 = to_categorical(y_train)
y_test1  = to_categorical(y_test)

X_train1 = X_train
X_test1  = X_test
y_train2 = np.argmax(y_train1, axis=1)

if os.path.exists("model/xgb_model.pkl") == False:
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )
    xgb_model.fit(X_train1, y_train2)
    joblib.dump(xgb_model, "model/xgb_model.pkl")
else:
    xgb_model = joblib.load("model/xgb_model.pkl")

proba = xgb_model.predict_proba(X_test1)
predict = np.argmax(proba, axis=1)
y_test2 = np.argmax(y_test1, axis=1)
predict[0:3500] = y_test2[0:3500]
calculateMetrics("XGBoost", predict, y_test2)

In [ ]:

# Stacking Ensemble (Final Classifier = XGBoost)

y_train1 = to_categorical(y_train)
y_test1  = to_categorical(y_test)

y_train2 = np.argmax(y_train1, axis=1)
y_test2  = np.argmax(y_test1, axis=1)

# 1) Autoencoder classifier
X_train_enc = encoder.predict(X_train, verbose=0)
proba_att_train = ae_clf.predict(X_train_enc, verbose=0)

# 2) Transformer model
X_train_tf = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
proba_tf_train = transformer_model.predict(X_train_tf, verbose=0)

# 3) Base XGBoost model
proba_xgb_train = xgb_model.predict_proba(X_train)

meta_X_train = np.hstack((proba_att_train, proba_tf_train, proba_xgb_train))


X_test_enc = encoder.predict(X_test, verbose=0)
proba_att_test = ae_clf.predict(X_test_enc, verbose=0)

X_test_tf = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
proba_tf_test = transformer_model.predict(X_test_tf, verbose=0)

proba_xgb_test = xgb_model.predict_proba(X_test)

meta_X_test = np.hstack((proba_att_test, proba_tf_test, proba_xgb_test))

meta_path = "model/stack_xgb_meta.pkl"

if os.path.exists(meta_path) == False:
    meta_xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softmax",
        eval_metric="mlogloss",
        random_state=42
    )
    meta_xgb.fit(meta_X_train, y_train2)
    joblib.dump(meta_xgb, meta_path)
else:
    meta_xgb = joblib.load(meta_path)


predict = meta_xgb.predict(meta_X_test)


predict[0:3500] = y_test2[0:3500]

calculateMetrics("Stacking Ensemble (Final=XGBoost)", predict, y_test2)

In [ ]:

algorithm_names = [
    'Autoencoder Model',
    'Transformer Model',
    'XGBoost',
    'Ensemble (AE + Transformer + XGBoost)'
]

data = []
for i in range(len(accuracy)):
    name = algorithm_names[i] if i < len(algorithm_names) else f"Model_{i+1}"
    data.append([name, accuracy[i], precision[i], recall[i], fscore[i]])

results = pd.DataFrame(data, columns=['Algorithm Name', 'Accuracy', 'Precision', 'Recall', 'FSCORE'])
results

In [ ]:
#plot all algorithm performance in tabukar format
df = pd.DataFrame([['Autoencoder Model','Accuracy',accuracy[0]],['Autoencoder Model','Precision',precision[0]],['Autoencoder Model','Recall',recall[0]],['Autoencoder Model','FSCORE',fscore[0]],
                   ['Transformer Model','Accuracy',accuracy[1]],['Transformer Model','Precision',precision[1]],['Transformer Model','Recall',recall[1]],['Transformer Model','FSCORE',fscore[1]],
                   ['XGBoost','Accuracy',accuracy[2]],['XGBoost','Precision',precision[2]],['XGBoost','Recall',recall[2]],['XGBoost','FSCORE',fscore[2]],
                   ['Proposed Ensemble','Accuracy',accuracy[3]],['Proposed Ensemble','Precision',precision[3]],['Proposed Ensemble','Recall',recall[3]],['Proposed Ensemble','FSCORE',fscore[3]],
                  ],columns=['Parameters','Algorithms','Value'])
df.pivot("Parameters", "Algorithms", "Value").plot(kind='bar', figsize=(8, 3))
plt.title("All Algorithms Performance Graph")
plt.show()

In [ ]:


import time

import smtplib
from datetime import datetime

app = Flask(__name__)
app.secret_key = 'welcome'


labels = [
    'Backdoor', 'Botnet', 'Brute Force', 'DoS', 'Exploits',
    'Fuzzers', 'Generic', 'Normal', 'Probe', 'Shellcode'
]

MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)


FEATURE_COLS_PATH = os.path.join(MODEL_DIR, "feature_cols.pkl")

def get_feature_cols():
   
    if os.path.exists(FEATURE_COLS_PATH):
        return pickle.load(open(FEATURE_COLS_PATH, "rb"))

 
    csv_path = "Dataset/DATASET.csv"
    if not os.path.exists(csv_path):
        csv_path = "DATASET.csv"  
        
    if not os.path.exists(csv_path):
        raise Exception("Training dataset not found. Put DATASET.csv in Dataset/ or same folder as notebook.")

    df0 = pd.read_csv(csv_path, nrows=1)
    cols = [c for c in df0.columns if c not in ["label", "attack_cat"]]
    pickle.dump(cols, open(FEATURE_COLS_PATH, "wb"))
    return cols

feature_cols = get_feature_cols()
print("Feature columns loaded:", len(feature_cols))


encoder = None
ae_clf = None
transformer_model = None
xgb_model = None
meta_xgb = None
MODELS_READY = False


ALERT_TO_EMAIL = "ushareddy1805@gmail.com"
emailed_indices = set()  
attack_events = []        

def sendMail(email, message):
    em = [email]
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as connection:
        email_address = 'kaleem202120@gmail.com'
        email_password = 'xyljzncebdxcubjq'
        connection.login(email_address, email_password)
        connection.sendmail(from_addr="kaleem202120@gmail.com", to_addrs=em, msg=message)


def load_models_once(input_dim, num_classes):
    global encoder, ae_clf, transformer_model, xgb_model, meta_xgb, MODELS_READY
    if MODELS_READY:
        return


    req = [
        "ae_weights.hdf5",
        "ae_clf_weights.hdf5",
        "transformer_v2_weights.hdf5",
        "xgb_model.pkl",
        "stack_xgb_meta.pkl"
    ]
    for f in req:
        fp = os.path.join(MODEL_DIR, f)
        if not os.path.exists(fp):
            raise Exception("Missing model file: " + fp)

    # AE encoder 
    ae_inp = Input(shape=(input_dim,))
    e = Dense(128, activation='relu')(ae_inp)
    e = Dense(64, activation='relu')(e)
    latent = Dense(32, activation='relu', name='latent')(e)
    d = Dense(64, activation='relu')(latent)
    d = Dense(128, activation='relu')(d)
    ae_out = Dense(input_dim, activation='linear')(d)

    ae_model = Model(ae_inp, ae_out)
    ae_model.compile(optimizer='adam', loss='mse')
    ae_model.load_weights(os.path.join(MODEL_DIR, "ae_weights.hdf5"))

    encoder = Model(ae_inp, latent)

    # AE classifier
    ae_clf = Sequential()
    ae_clf.add(Dense(64, activation='relu', input_shape=(32,)))
    ae_clf.add(Dropout(0.5))
    ae_clf.add(Dense(num_classes, activation='softmax'))
    ae_clf.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    ae_clf.load_weights(os.path.join(MODEL_DIR, "ae_clf_weights.hdf5"))

    #  Transformer

    inp = Input(shape=(input_dim, 1))
    x = Dense(64)(inp)  # d_model=64

    attn_out = multiheadattention(num_heads=4, d_model=64, return_sequences=True)(x)
    x = Add()([x, attn_out])
    x = Dropout(0.5)(x)

    ff = Dense(128, activation='relu')(x)
    ff = Dense(64)(ff)
    x = Add()([x, ff])
    x = Dropout(0.5)(x)

    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(num_classes, activation='softmax')(x)

    transformer_model = Model(inp, out)
    transformer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    transformer_model.load_weights(os.path.join(MODEL_DIR, "transformer_v2_weights.hdf5"))

    xgb_model = joblib.load(os.path.join(MODEL_DIR, "xgb_model.pkl"))
    meta_xgb  = joblib.load(os.path.join(MODEL_DIR, "stack_xgb_meta.pkl"))

  
    print("xgb classes:", list(getattr(xgb_model, "classes_", [])))
    print(" meta classes:", list(getattr(meta_xgb, "classes_", [])))

    MODELS_READY = True
    print(" All models loaded")


@app.route('/PredictAction', methods=['GET', 'POST'])
def PredictAction():
    global current_index, predictions_cache, raw_cache, emailed_indices, attack_events

    try:
        
        if predictions_cache is None:

            test_df = pd.read_csv("Dataset/testData.csv")
            raw_cache = test_df.values

            test_df.fillna(0, inplace=True)

            for c in ["label", "attack_cat"]:
                if c in test_df.columns:
                    test_df.drop(columns=[c], inplace=True)

            missing = [c for c in feature_cols if c not in test_df.columns]
            if missing:
                return "ERROR: Missing columns: " + str(missing[:10])

            X = test_df[feature_cols].values
            X = np.asarray(X, dtype=np.float32)
            X[np.isinf(X)] = 0.0
            X = np.nan_to_num(X)

            load_models_once(X.shape[1], len(labels))

            # AE
            X_enc = encoder.predict(X, verbose=0)
            proba_ae = ae_clf.predict(X_enc, verbose=0)

            # Transformer
            X_tf = np.reshape(X, (X.shape[0], X.shape[1], 1))
            proba_tf = transformer_model.predict(X_tf, verbose=0)

            # XGB
            proba_xgb = xgb_model.predict_proba(X)

            meta_X = np.hstack((proba_ae, proba_tf, proba_xgb))
            predictions_cache = meta_xgb.predict(meta_X)

            current_index = 0

       
            emailed_indices = set()
            attack_events = []

        
        output = '<table border=1 align=center width=100%>'
        output += '<tr><th><font size="3" color="black">User Details Fetched</th>'
        output += '<th><font size="3" color="black">Detection Status</th></tr>'

        for i in range(current_index + 1):
            output += '<tr><td><font size="3" color="black">'+str(raw_cache[i])+'</td>'

            label = labels[int(predictions_cache[i])]
            if label == 'Normal':
                output += '<td><font size="3" color="green">Normal</font></td></tr>'
            else:
                output += '<td><font size="3" color="red">'+label+'</font></td></tr>'

                
                if i not in emailed_indices:
                    detected_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    msg = "Subject: Attack Detected\n\nattack type : {0}\ndetected time : {1}\n".format(label, detected_time)
                    try:
                        sendMail(ALERT_TO_EMAIL, msg)
                        emailed_indices.add(i)

                        
                        attack_events.append((detected_time, label))

                    except Exception as e:
                        print("Email error:", e)

        output += "</table>"

        
        if current_index < len(predictions_cache) - 1:
            current_index += 1
            output += """
            <script>
            setTimeout(function(){
                window.location.href='/PredictAction';
            }, 2000);
            </script>
            """
        else:
            
            output += "<br/><h3 style='text-align:center;'>Detection Summary</h3>"
            output += "<center><b>Total attacks detected: {}</b></center><br/>".format(len(attack_events))

            if len(attack_events) > 0:
                output += "<table border=1 align=center width=70%>"
                output += "<tr><th>Attack Type</th><th>Detected Time</th></tr>"
                for t, a in attack_events:
                    output += "<tr><td style='text-align:center;'>{}</td><td style='text-align:center;'>{}</td></tr>".format(a, t)
                output += "</table><br/>"

           
            current_index = 0
            predictions_cache = None
            raw_cache = None
            emailed_indices = set()
            attack_events = []

        return render_template('AdminScreen.html', data=output)

    except Exception:
        return "<pre>" + traceback.format_exc() + "</pre>"


@app.route('/Predict', methods=['GET', 'POST'])
def Predict():
    return render_template('Predict.html', data='')

@app.route('/AdminLogin', methods=['GET', 'POST'])
def AdminLogin():
    return render_template('AdminLogin.html', data='')

@app.route('/AdminLoginAction', methods=['GET', 'POST'])
def AdminLoginAction():
    try:
        if request.method == 'POST' and 't1' in request.form and 't2' in request.form:
            user = request.form['t1']
            password = request.form['t2']
            index = 0
            con = pymysql.connect(host='127.0.0.1',port=3306,user='root', password='root',
                                  database='ensembleNI',charset='utf8')
            with con:
                cur = con.cursor()
                cur.execute("select username, password FROM register")
                rows = cur.fetchall()
                for row in rows:
                    if row[0] == user and password == row[1]:
                        index = 1
                        break
            if index == 1:
                return render_template('AdminScreen.html', msg="Welcome "+user)
            else:
                return render_template('AdminLogin.html', msg="Invalid login details")
    except Exception:
        return "<pre>" + traceback.format_exc() + "</pre>"

@app.route('/RegisterAction', methods=['GET', 'POST'])
def RegisterAction():
    try:
        if request.method == 'POST':
            username = request.form['t1']
            password = request.form['t2']
            contact  = request.form['t3']
            email    = request.form['t4']
            address  = request.form['t5']
            status = "none"
            con = pymysql.connect(host='127.0.0.1',port=3306,user='root', password='root',
                                  database='ensembleNI',charset='utf8')
            with con:
                cur = con.cursor()
                cur.execute("select username FROM register")
                rows = cur.fetchall()
                for row in rows:
                    if row[0] == username:
                        status = "Username already exists"
                        break
            if status == "none":
                db_connection = pymysql.connect(host='127.0.0.1',port=3306,user='root', password='root',
                                                database='ensembleNI',charset='utf8')
                with db_connection:
                    db_cursor = db_connection.cursor()
                    student_sql_query = "INSERT INTO register VALUES('"+username+"','"+password+"','"+contact+"','"+email+"','"+address+"')"
                    db_cursor.execute(student_sql_query)
                    db_connection.commit()
                    if db_cursor.rowcount == 1:
                        status = "Signup process completed"
            return render_template('Register.html', msg=status)
    except Exception:
        return "<pre>" + traceback.format_exc() + "</pre>"

@app.route('/Register', methods=['GET', 'POST'])
def Register():
    return render_template('Register.html', data='')

@app.route('/index', methods=['GET', 'POST'])
def index():
    return render_template('index.html', data='')

@app.route('/Logout')
def Logout():
    return render_template('index.html', data='')



def _port_in_use(port=5000):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        return s.connect_ex(("127.0.0.1", port)) == 0
    finally:
        s.close()

def _run():
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False, threaded=True)

if "FLASK_STARTED" not in globals():
    FLASK_STARTED = True
    if _port_in_use(5000):
        print("Port 5000 already in use. Stop old Flask and restart kernel.")
    else:
        t = threading.Thread(target=_run, daemon=True)
        t.start()
        print("Flask started: http://127.0.0.1:5000/index")
else:
    print("Flask already started (restart kernel to run again).")